# NYC TLC: exploration

A narrated pass over the medallion tables: overview, month coverage, data quality, and the four business questions, with charts. It connects through Databricks Connect, reusing the pipeline's `get_spark()`, so it talks to the same serverless workspace.

It reads the `nyc_tlc_dev` catalog by default (the pipeline default). Set `NYC_TLC_CATALOG=nyc_tlc` to point at prod. Every query here aggregates in Spark and only brings small results back with `.toPandas()`, never a full table.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# reuse the pipeline's session helper so the notebook connects exactly like the pipeline
sys.path.insert(0, str(Path.cwd().parents[1] / "src" / "pipeline"))
from config import CATALOG, get_spark  # noqa: E402

spark = get_spark()
spark.sql(f"USE CATALOG {CATALOG}")


def q(sql: str) -> pd.DataFrame:
    """Run SQL and return a pandas DataFrame (results are already aggregated and small)."""
    return spark.sql(sql).toPandas()


print(f"connected, catalog = {CATALOG}")

## 1. Overview

Row counts down the layers for yellow. Conformation is 1:1, so bronze, silver and gold must match.

In [ ]:
q("""
  select 'bronze' as layer, count(*) as rows from bronze.yellow_tripdata_raw
  union all select 'silver', count(*) from silver.taxi_trips where taxi_type = 'yellow'
  union all select 'gold', count(*) from gold.obt_trips where taxi_type = 'yellow'
""")

## 2. Coverage

Which months landed, per taxi type. Year and month are parsed from the source file name, so this is also a quick ingestion sanity check.

In [ ]:
coverage = q("""
  select taxi_type, month, count(*) as trips
  from gold.obt_trips
  where year = 2023
  group by taxi_type, month
  order by month
""")
coverage.pivot_table(index="month", columns="taxi_type", values="trips").plot(kind="bar")
plt.title("Trips per month by taxi type (2023)")
plt.xlabel("month")
plt.ylabel("trips")
plt.tight_layout()
plt.show()

## 3. Data quality

Negative `total_amount` is kept on purpose (refunds and voids) and flagged `is_amount_valid` in silver. Here is the split.

In [ ]:
q("""
  select is_amount_valid, count(*) as rows,
    min(total_amount) as min_amount, max(total_amount) as max_amount
  from silver.taxi_trips
  group by is_amount_valid
""")

## 4. Business questions

All four answers come from gold (`obt_trips`); the Jan to May 2023 scope lives here, in the query.

### Q1: average total_amount per month, yellow only

In [ ]:
q1 = q("""
  select month, round(avg(total_amount), 2) as avg_total_amount
  from gold.obt_trips
  where taxi_type = 'yellow' and year = 2023 and month between 1 and 5
  group by month
  order by month
""")
plt.plot(q1["month"], q1["avg_total_amount"], marker="o")
plt.title("Q1: avg total_amount per month (yellow)")
plt.xlabel("month")
plt.ylabel("avg total_amount")
plt.tight_layout()
plt.show()

### Q2: average passengers per hour, May 2023, all taxis

In [ ]:
q2 = q("""
  select pickup_hour, round(avg(passenger_count), 2) as avg_passengers
  from gold.obt_trips
  where year = 2023 and month = 5 and passenger_count > 0
  group by pickup_hour
  order by pickup_hour
""")
plt.plot(q2["pickup_hour"], q2["avg_passengers"], marker="o")
plt.title("Q2: avg passengers per hour (May 2023)")
plt.xlabel("hour")
plt.ylabel("avg passengers")
plt.tight_layout()
plt.show()

### Q3: revenue and average ticket per taxi type and month

In [ ]:
q3 = q("""
  select taxi_type, month, round(sum(total_amount), 2) as total_revenue
  from gold.obt_trips
  where year = 2023 and month between 1 and 5
  group by taxi_type, month
  order by month
""")
q3.pivot_table(index="month", columns="taxi_type", values="total_revenue").plot(kind="bar")
plt.title("Q3: total revenue per month by taxi type")
plt.xlabel("month")
plt.ylabel("total revenue")
plt.tight_layout()
plt.show()

### Q4: trip volume by hour of day, May 2023

In [ ]:
q4 = q("""
  select pickup_hour, count(*) as trips
  from gold.obt_trips
  where year = 2023 and month = 5
  group by pickup_hour
  order by pickup_hour
""")
plt.bar(q4["pickup_hour"], q4["trips"])
plt.title("Q4: trip volume by hour (May 2023)")
plt.xlabel("hour")
plt.ylabel("trips")
plt.tight_layout()
plt.show()